# Model comparison

In [2]:
from src.transformer_absa import TransformerABSA
from src.lexicon_absa import LexiconABSA
from src.llm_absa import LLMABSA
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import time

def extract_labels_from_absa_outputs(absa_outputs):
    """Convert [(aspect, sentiment), ...] into {aspect: sentiment} dict."""
    label_dict = {}
    for aspect, sentiment in absa_outputs:
        label_dict[aspect.lower()] = sentiment.lower()
    return label_dict

def extract_labels_and_confidence_from_llm_outputs(llm_outputs):
    """
    Convert list of LLM AspectSentiment objects into:
    - label_dict: {aspect: sentiment}
    - conf_dict: {aspect: confidence}
    """
    label_dict = {}
    conf_dict = {}
    for item in llm_outputs:
        aspect = item.aspect.lower()
        sentiment = item.sentiment.lower()
        confidence = item.confidence
        label_dict[aspect] = sentiment
        conf_dict[aspect] = confidence
    return label_dict, conf_dict

def align_labels(true_outputs, pred_outputs):
    """Align true and predicted labels lists for metric evaluation."""
    true_dict = extract_labels_from_absa_outputs(true_outputs)
    pred_dict = extract_labels_from_absa_outputs(pred_outputs)
    y_true, y_pred = [], []
    for aspect in true_dict:
        y_true.append(true_dict[aspect])
        y_pred.append(pred_dict.get(aspect, 'missing'))
    return y_true, y_pred

def evaluate_predictions(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    return precision, recall, f1, accuracy

def average_confidence(conf_dict, aspects):
    confidences = [conf_dict.get(a.lower(), 0) for a in aspects]
    return sum(confidences) / len(confidences) if confidences else 0.0

def run_evaluation_on_model(model, test_data, is_llm=False):
    all_true, all_pred = [], []
    confidences = []
    total_time = 0.0
    
    for text, true_labels in test_data:
        start = time.time()
        
        pred_outputs = model.analyze(text)

        print(f"Output type for model {model.__class__.__name__}: {type(pred_outputs[0]) if pred_outputs else 'empty'}")
        
        if is_llm:
            # Should be list of AspectSentiment objects
            pred_dict, conf_dict = extract_labels_and_confidence_from_llm_outputs(pred_outputs)
        else:
            # Should be list of tuples
            # Safe fallback if pred_outputs are objects, convert or error here
            if isinstance(pred_outputs[0], tuple):
                pred_dict = extract_labels_from_absa_outputs(pred_outputs)
            else:
                # Convert objects with aspect and sentiment attributes to tuples
                try:
                    pred_tuples = [(obj.aspect, obj.sentiment) for obj in pred_outputs]
                    pred_dict = extract_labels_from_absa_outputs(pred_tuples)
                except Exception as e:
                    print(f"Error converting prediction outputs: {e}")
                    raise
        
            conf_dict = {}
        
        elapsed = time.time() - start
        total_time += elapsed
        
        true_dict = extract_labels_from_absa_outputs(true_labels)
        
        for aspect in true_dict:
            all_true.append(true_dict[aspect])
            all_pred.append(pred_dict.get(aspect, 'missing'))
            if is_llm:
                confidences.append(conf_dict.get(aspect, 0.0))
    
    precision, recall, f1, accuracy = evaluate_predictions(all_true, all_pred)
    avg_conf = average_confidence(dict(zip([a.lower() for a in true_dict.keys()], confidences)), true_dict.keys()) if confidences else None
    
    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "accuracy": accuracy,
        "avg_confidence": avg_conf,
        "avg_inference_time": total_time / len(test_data) if len(test_data) > 0 else 0.0,
    }



lexicon_model = LexiconABSA()
transformer_model = TransformerABSA()
llm_model = LLMABSA()


test_data = [
    ("The hotel room was spacious and clean, but the breakfast was disappointing.", 
     [('hotel room', 'positive'), ('breakfast', 'negative')]),
    # We can add more labeled test sentences if we want
]

results = {}
results["Lexicon"] = run_evaluation_on_model(lexicon_model, test_data)
results["Transformer"] = run_evaluation_on_model(transformer_model, test_data)
results["LLM"] = run_evaluation_on_model(llm_model, test_data, is_llm=True)

for name, metrics in results.items():
    print(f"Results for {name}:")
    print(f" Precision: {metrics['precision']:.3f}")
    print(f" Recall: {metrics['recall']:.3f}")
    print(f" F1-score: {metrics['f1']:.3f}")
    print(f" Accuracy: {metrics['accuracy']:.3f}")
    if metrics['avg_confidence'] is not None:
        print(f" Avg Confidence: {metrics['avg_confidence']:.3f}")
    print(f" Avg Inference Time (s): {metrics['avg_inference_time']:.3f}\n")


C:\Users\khale\Desktop\KDG\year3\data6\data_6_llm_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\khale\Desktop\KDG\year3\data6\data_6_llm_project\.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:515: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Output type for model LexiconABSA: <class 'src.base.AspectSentiment'>
Output type for model TransformerABSA: <class 'tuple'>
Output type for model LLMABSA: <class 'src.base.AspectSentiment'>
Results for Lexicon:
 Precision: 1.000
 Recall: 1.000
 F1-score: 1.000
 Accuracy: 1.000
 Avg Inference Time (s): 0.158

Results for Transformer:
 Precision: 0.500
 Recall: 0.500
 F1-score: 0.500
 Accuracy: 0.500
 Avg Inference Time (s): 0.263

Results for LLM:
 Precision: 1.000
 Recall: 1.000
 F1-score: 1.000
 Accuracy: 1.000
 Avg Confidence: 0.900
 Avg Inference Time (s): 58.854



The LexiconABSA and LLMABSA models both demonstrated perfect accuracy, precision, recall, and F1-score on the test data, indicating strong performance in aspect-based sentiment analysis. However, the LLMABSA model required significantly more inference time (around 59 seconds) compared to the LexiconABSA (0.16 seconds) and TransformerABSA (0.26 seconds). The TransformerABSA model showed moderate performance with a 50% accuracy and F1-score on this test, suggesting room for improvement or further tuning.

In summary, the lexicon-based approach offers fast and reliable results with minimal resource use, while the LLM-based approach achieves excellent accuracy but with much longer processing time due to resource demands. The transformer-based method falls in between but may require optimization to improve its accuracy. Depending on the use case, one might prioritize speed and efficiency or accuracy with higher resource requirements.